# Assignment 2: Stochastic Gradient Descent (SGD) on Audi used-car data
Regression: predict `price` from car features (N = 101,668 data points).
Loss: Mean Squared Error (MSE).

**Strategy-1**: random sampling *with* replacement, `ceil(N/bs)` updates per epoch.
**Strategy-2**: random permutation each epoch (*without* replacement), split in batches, `ceil(N/bs)` updates per epoch.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from math import ceil
np.random.seed(42)
df = pd.read_csv('audi.csv')
print(df.shape)
df.head()

## 1. Preprocessing
One-hot encode categorical columns, standardise features and target (helps SGD converge), add bias column, 80/20 train-test split.

In [ ]:
df = pd.get_dummies(df, columns=['model','transmission','fuelType'], drop_first=True)
y = df['price'].values.astype(float)
X = df.drop(columns='price').values.astype(float)

idx = np.random.permutation(len(X))
split = int(0.8*len(X))
tr, te = idx[:split], idx[split:]
Xtr, Xte, ytr, yte = X[tr], X[te], y[tr], y[te]

mu, sd = Xtr.mean(0), Xtr.std(0); sd[sd==0] = 1
Xtr = (Xtr-mu)/sd; Xte = (Xte-mu)/sd
ymu, ysd = ytr.mean(), ytr.std()
ytr_s = (ytr-ymu)/ysd

# add bias column
Xtr = np.hstack([np.ones((len(Xtr),1)), Xtr])
Xte = np.hstack([np.ones((len(Xte),1)), Xte])
N, d = Xtr.shape
print('Train N =', N, ' features =', d)

## 2. Helper functions

In [ ]:
def mse(X, y, w):
    r = X@w - y
    return np.mean(r**2)

def grad(Xb, yb, w):
    # gradient of (1/B) * sum (x^T w - y)^2
    return (2/len(yb)) * Xb.T @ (Xb@w - yb)

def test_rmse_price(w):
    pred = Xte@w*ysd + ymu
    return np.sqrt(np.mean((pred-yte)**2))

## 3. SGD with the two strategies

In [ ]:
def sgd(X, y, bs, epochs=100, lr=0.002, strategy=1, seed=0):
    rng = np.random.default_rng(seed)
    N, d = X.shape
    w = np.zeros(d)
    steps = ceil(N/bs)
    hist = []
    for ep in range(epochs):
        if strategy == 1:                       # random WITH replacement
            for _ in range(steps):
                b = rng.integers(0, N, size=bs)  # uniform random, repeats allowed
                w -= lr*grad(X[b], y[b], w)
        else:                                   # random WITHOUT replacement
            perm = rng.permutation(N)
            for s in range(steps):
                b = perm[s*bs:(s+1)*bs]
                w -= lr*grad(X[b], y[b], w)
        hist.append(mse(X, y, w))               # full train MSE after each epoch
    return w, hist

## 4. Run: 5 batch sizes x 2 strategies, 100 epochs

In [ ]:
batch_sizes = [32, 64, 128, 512, 2048]
epochs, lr = 100, 0.002
results, hists = [], {}
for strat in (1, 2):
    for bs in batch_sizes:
        t = time.time()
        w, h = sgd(Xtr, ytr_s, bs, epochs, lr, strat)
        hists[(strat, bs)] = h
        results.append(dict(strategy=strat, batch_size=bs, updates_per_epoch=ceil(N/bs),
                            final_train_MSE=h[-1], test_RMSE_price=test_rmse_price(w),
                            time_s=round(time.time()-t,1)))
        print(results[-1])
res = pd.DataFrame(results); res

## 5. Convergence plots

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14,5), sharey=True)
for s, a in zip((1,2), ax):
    for bs in batch_sizes:
        a.plot(hists[(s,bs)], label=f'bs={bs}')
    a.set_yscale('log'); a.set_xlabel('Epoch'); a.set_ylabel('Train MSE (scaled)')
    a.set_title('Strategy-1: with replacement' if s==1 else 'Strategy-2: without replacement')
    a.legend()
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
for s in (1,2):
    ax.plot(batch_sizes, [hists[(s,b)][-1] for b in batch_sizes], 'o-', label=f'Strategy-{s}')
ax.set_xscale('log', base=2); ax.set_xlabel('Batch size'); ax.set_ylabel('Final train MSE'); ax.legend(); plt.show()

## 6. Sanity check vs closed-form least squares

In [ ]:
w_ls = np.linalg.lstsq(Xtr, ytr_s, rcond=None)[0]
print('Closed-form train MSE:', mse(Xtr, ytr_s, w_ls), ' test RMSE (price):', test_rmse_price(w_ls))

## 7. Observations
*(Apni nijer result dekhe likhun, e.g.)*
- Choto batch size => beshi noisy but epoch-e kom time e beshi update, tai tara tari converge kore.
- Boro batch size => smooth curve kintu per-epoch update kom.
- Strategy-2 (without replacement) usually ektu stable / shob data point proti epoch-e exactly ekbar use hoy.
- Final MSE closed-form solution er kachakachi.